# Capítulo 7 — Computação Numérica com NumPy

**Programação com Python Aplicada à Engenharia de Defesa** · IPETEC/UCP

> **Notebook de aula — Módulo III, Sábado 3.** Ao final, o miniprojeto ganha **rotinas de cálculo numérico** — estatísticas e detecção de contatos atípicos (conclusão do Módulo 3).

---

### Objetivos do capítulo
Ao final, você será capaz de:
- criar e inspecionar **arrays** do NumPy, uni e multidimensionais;
- aplicar **operações vetorizadas** a conjuntos inteiros de dados, sem laços;
- filtrar dados com **indexação** e **máscaras booleanas**;
- resumir dados com **agregações** estatísticas (média, desvio, percentis);
- operar com **matrizes** e noções de **álgebra linear** aplicada;
- acrescentar ao **miniprojeto** rotinas de cálculo numérico.

*No Capítulo 3, processávamos listas de leituras com laços. Quando o volume cresce — milhares de leituras —, os laços em Python ficam lentos e verbosos. O **NumPy** resolve os dois problemas: oferece o `array` e permite operar sobre ele de forma **vetorizada**, aplicando um cálculo a todos os elementos de uma vez, em código otimizado.*

> 📝 **Nota** — O NumPy é a fundação do ecossistema científico do Python (pandas, Matplotlib e as bibliotecas de *machine learning* são construídas sobre arrays NumPy). Ele já vem instalado no Google Colab — nada a instalar.

## 7.1 O array NumPy

O **array** é o objeto central do NumPy: uma sequência de elementos do *mesmo tipo*, armazenada de forma compacta e eficiente. A convenção universal é importar o NumPy com o apelido `np`.

**Listagem 7.1 — Criando arrays.**

In [ ]:
import numpy as np

# A partir de uma lista
leituras = np.array([28.4, 33.1, 41.7, 39.0, 45.2])
print(leituras)            # [28.4 33.1 41.7 39.  45.2]
print(leituras.shape)      # (5,)      -> forma: 5 elementos
print(leituras.dtype)      # float64   -> tipo dos elementos

# Geradores úteis
print(np.zeros(3))         # [0. 0. 0.]
print(np.arange(0, 10, 2)) # [0 2 4 6 8]  (início, fim, passo)
print(np.linspace(0, 90, 5))  # 5 valores igualmente espaçados de 0 a 90

Dois atributos são fundamentais. O `shape` descreve a **forma** do array (quantos elementos, em quantas dimensões); o `dtype`, o tipo dos elementos (aqui, `float64`). Diferentemente de uma lista, todos os elementos de um array compartilham o mesmo tipo — o que permite ao NumPy armazená-los e processá-los com grande eficiência.

## 7.2 Operações vetorizadas

Aqui está a ideia que muda tudo. Em vez de percorrer os elementos com um laço, aplicamos a operação ao array inteiro de uma só vez. Lembra a conversão de nós para km/h do Capítulo 2? Agora ela se faz sobre todas as leituras com uma única expressão.

**Listagem 7.2 — Conversão vetorizada: todas as leituras de uma vez.**

In [ ]:
import numpy as np

leituras_no = np.array([12, 18, 24, 9, 27])

# Sem laço: a multiplicação se aplica a cada elemento
leituras_kmh = leituras_no * 1.852
print(leituras_kmh)        # [22.224 33.336 44.448 16.668 50.004]

Operações entre dois arrays também são elemento a elemento, e o NumPy traz um arsenal de **funções universais** — raiz, seno, exponencial — que operam da mesma forma vetorizada.

**Listagem 7.3 — Operações elemento a elemento e funções universais.**

In [ ]:
import numpy as np

a = np.array([1, 2, 3])
b = np.array([10, 20, 30])
print(a + b)                    # [11 22 33]
print(a * b)                    # [10 40 90]
print(np.sqrt(np.array([1, 4, 9])))   # [1. 2. 3.]

> ✅ **Boa prática** — Prefira operações vetorizadas a laços sempre que trabalhar com arrays. Além de mais concisas e legíveis, são muito mais rápidas: o NumPy executa o cálculo em código otimizado, não no laço interpretado do Python. Para grandes volumes de dados de sensores, a diferença pode ser de dezenas a centenas de vezes. A regra prática: se você escreveu um laço para percorrer um array, pergunte-se se não há uma expressão vetorizada equivalente.

Como aplicação de engenharia, retomemos o alcance balístico do Capítulo 2. Para descobrir o alcance de um projétil em *vários* ângulos de lançamento, calculamos todos de uma vez.

**Listagem 7.4 — Alcance para vários ângulos, vetorizado.**

In [ ]:
import numpy as np

v = 50.0                                  # velocidade (m/s)
g = 9.81
angulos = np.array([15, 30, 45, 60, 75])  # graus

rad = np.radians(angulos)                 # converte todos para radianos
alcances = v**2 * np.sin(2 * rad) / g     # fórmula aplicada a todos

print(np.round(alcances, 1))              # [127.4 220.7 254.8 220.7 127.4]

O resultado confirma um fato conhecido da balística: o alcance é máximo a 45° e simétrico em torno desse ângulo. Toda a tabela emergiu de **uma única linha** de cálculo.

## 7.3 Indexação, fatiamento e máscaras booleanas

Arrays são indexados e fatiados como as listas do Capítulo 3 — mesma notação de colchetes, índices a partir de zero. A novidade poderosa do NumPy é a **máscara booleana**: uma comparação aplicada a um array produz um array de valores lógicos, que pode então *selecionar* elementos.

**Listagem 7.5 — Filtrando com máscara booleana.**

In [ ]:
import numpy as np

leituras = np.array([28.4, 33.1, 41.7, 39.0, 45.2])
limite = 40.0

# A comparação gera uma máscara de True/False
mascara = leituras > limite
print(mascara)              # [False False  True False  True]

# A máscara seleciona apenas os elementos correspondentes a True
print(leituras[mascara])    # [41.7 45.2]

# Tudo em uma linha, idiomático:
print(leituras[leituras > limite])   # [41.7 45.2]

Compare `leituras[leituras > limite]` com o laço e o `if` que escreveríamos no Capítulo 2 para o mesmo fim. A máscara booleana exprime "os elementos que satisfazem esta condição" de forma direta — e é, novamente, vetorizada e veloz.

## 7.4 Agregações e estatística

Resumir um conjunto de dados em poucos números — média, dispersão, extremos — é o primeiro passo de qualquer análise. O NumPy traz essas agregações como métodos do próprio array.

**Listagem 7.6 — Estatísticas de um array.**

In [ ]:
import numpy as np

leituras = np.array([28.4, 33.1, 41.7, 39.0, 45.2])

print("Média:", leituras.mean())            # 37.48
print("Desvio padrão:", leituras.std())     # 6.0224...
print("Máximo:", leituras.max())            # 45.2
print("Mínimo:", leituras.min())            # 28.4
print("Percentil 90:", np.percentile(leituras, 90))   # 43.8

O **desvio padrão** mede a dispersão dos dados em torno da média. O **percentil 90** é o valor abaixo do qual estão 90% das leituras — uma forma robusta de caracterizar "o que é alto" sem se deixar enganar por um único valor extremo. São ferramentas que usaremos para distinguir o normal do anômalo no miniprojeto.

> ⚠️ **Armadilha comum** — O `array` exige um único `dtype`. Ao misturar números e texto, o NumPy converte **tudo para texto**, silenciosamente — e as contas param de funcionar. Veja:

In [ ]:
import numpy as np

misturado = np.array([1, 2, "alerta"])
print(misturado.dtype)      # <U21  -> virou texto (Unicode)!
print(misturado)            # ['1' '2' 'alerta']
# misturado.mean()          # provocaria erro: não há média de texto

## 7.5 Arrays multidimensionais e álgebra linear

Um array pode ter mais de uma dimensão. Um array bidimensional é, na prática, uma **matriz** — uma tabela de números. A forma (`shape`) passa a ter dois valores: linhas e colunas.

**Listagem 7.7 — Um array bidimensional e agregações por eixo.**

In [ ]:
import numpy as np

# Duas linhas, três colunas (p. ex., 2 sensores, 3 leituras cada)
M = np.array([[1, 2, 3],
              [4, 5, 6]])

print(M.shape)            # (2, 3)
print(M.mean(axis=0))     # [2.5 3.5 4.5]  média de cada coluna
print(M.mean(axis=1))     # [2. 5.]        média de cada linha

O parâmetro `axis` controla a direção da agregação: `axis=0` opera ao longo das linhas (um valor por coluna), e `axis=1`, ao longo das colunas (um valor por linha). Esse conceito reaparecerá, idêntico, no pandas do próximo capítulo.

### 7.5.1 Operações de álgebra linear
O NumPy oferece as operações clássicas da álgebra linear, base de muitos cálculos de engenharia. A **norma** de um vetor dá o seu comprimento; aplicada à diferença entre duas posições, fornece a distância entre elas.

**Listagem 7.8 — Distância entre duas posições.**

In [ ]:
import numpy as np

base = np.array([0.0, 0.0])
contato = np.array([3.0, 4.0])

distancia = np.linalg.norm(contato - base)
print(distancia)          # 5.0  (o clássico triângulo 3-4-5)

A multiplicação de matrizes, com o operador `@`, permite aplicar **transformações** a coordenadas. Uma matriz de rotação, por exemplo, gira um ponto em torno da origem.

**Listagem 7.9 — Rotação de um ponto por uma matriz.**

In [ ]:
import numpy as np

# Matriz de rotação de 90 graus no sentido anti-horário
R = np.array([[0, -1],
              [1,  0]])
ponto = np.array([2, 0])

print(R @ ponto)          # [0 2]  -> o ponto girou 90 graus

Por fim, `np.linalg.solve` resolve um sistema de equações lineares $A\mathbf{x} = \mathbf{b}$ — situação comum em problemas de equilíbrio, ajuste e estimação.

**Listagem 7.10 — Resolvendo um sistema linear.**

In [ ]:
import numpy as np

A = np.array([[2.0, 1.0],
              [1.0, 3.0]])
b = np.array([1.0, 2.0])

x = np.linalg.solve(A, b)   # resolve A x = b
print(x)                    # [0.2 0.6]

> 🛡️ **Contexto de defesa** — Operações de álgebra linear estão no coração de muitas tecnologias estratégicas. A fusão de dados de múltiplos sensores, a estimação de trajetórias, o processamento de sinais e os próprios algoritmos de *Machine Learning* reduzem-se, em larga medida, a multiplicações de matrizes e à resolução de sistemas. O NumPy é a ferramenta que torna esses cálculos eficientes em Python — e a base sobre a qual as bibliotecas mais avançadas se erguem.

## 7.6 Miniprojeto: rotinas de cálculo numérico (Módulo 3)

Concluímos o Módulo 3 dotando o sistema de **análise numérica**. Acrescentaremos à classe `RegistroDeOcorrencias` (do Capítulo 5) métodos que, com o NumPy, resumem as velocidades registradas e identificam contatos estatisticamente atípicos.

Para que o notebook rode de forma autônoma, a célula abaixo inclui a classe **completa** — os atributos e métodos do Capítulo 5 (que o livro resume com `# ...`) **mais** as novas rotinas de cálculo.

**Listagem 7.11 — Miniprojeto, Módulo 3 (conclusão): rotinas de cálculo com NumPy.**

In [ ]:
import numpy as np


class Ocorrencia:
    """Uma ocorrência de monitoramento (Capítulo 5)."""

    def __init__(self, id, sensor, velocidade_kmh, tipo="superfície"):
        self.id = id
        self.sensor = sensor
        self.velocidade_kmh = velocidade_kmh
        self.tipo = tipo

    def em_alerta(self, limite=40.0):
        return self.velocidade_kmh > limite

    def __repr__(self):
        return f"Ocorrencia(#{self.id}, {self.sensor}, {self.velocidade_kmh} km/h)"


class RegistroDeOcorrencias:
    # ---- atributos e métodos do Capítulo 5 ----
    def __init__(self, limite=40.0):
        self._ocorrencias = []
        self.limite = limite

    def registrar(self, sensor, velocidade_kmh, tipo="superfície"):
        novo_id = len(self._ocorrencias) + 1
        ocorrencia = Ocorrencia(novo_id, sensor, velocidade_kmh, tipo)
        self._ocorrencias.append(ocorrencia)
        return ocorrencia

    def alertas(self):
        return [o for o in self._ocorrencias if o.em_alerta(self.limite)]

    def total(self):
        return len(self._ocorrencias)

    # ---- novas rotinas de cálculo (Capítulo 7) ----
    def velocidades(self):
        """Devolve as velocidades das ocorrências como um array NumPy."""
        return np.array([o.velocidade_kmh for o in self._ocorrencias])

    def estatisticas(self):
        """Resume as velocidades: média, desvio, máxima e percentil 90."""
        v = self.velocidades()
        if len(v) == 0:
            return None
        return {
            "media": v.mean(),
            "desvio": v.std(),
            "maxima": v.max(),
            "p90": np.percentile(v, 90),
        }

    def anomalias(self, n_desvios=2.0):
        """Ocorrências cuja velocidade excede média + n desvios padrão."""
        v = self.velocidades()
        if len(v) == 0:
            return []
        limiar = v.mean() + n_desvios * v.std()
        return [o for o in self._ocorrencias if o.velocidade_kmh > limiar]

Suponha um registro com seis ocorrências, uma delas — um contato a 120 km/h — nitidamente fora do padrão. As novas rotinas a flagram.

**Listagem 7.12 — Usando as rotinas de cálculo.**

In [ ]:
registro = RegistroDeOcorrencias()
registro.registrar("Radar-A1", 22.2)
registro.registrar("Radar-A1", 44.4)
registro.registrar("Sonar-1", 18.5)
registro.registrar("Radar-B2", 50.0)
registro.registrar("Radar-B2", 33.1)
registro.registrar("Radar-A1", 120.0)    # contato atípico

est = registro.estatisticas()
print(f"Média: {est['media']:.1f} km/h")
print(f"Desvio padrão: {est['desvio']:.1f} km/h")
print(f"Percentil 90: {est['p90']:.1f} km/h")

print("Contatos atípicos:")
for o in registro.anomalias(n_desvios=2.0):
    print(f"  #{o.id} {o.sensor}: {o.velocidade_kmh} km/h")

A regra "média mais dois desvios padrão" é uma forma simples e consagrada de detectar valores atípicos: aponta o que se afasta demais do comportamento usual do conjunto. Note que a análise vive em métodos do *registro* — pois opera sobre a coleção —, respeitando a divisão de responsabilidades do Capítulo 5. O sistema, que já registrava, persistia e exibia ocorrências, agora também as **analisa**.

> 📝 **Nota** — Com isto, encerra-se o **Módulo 3**. O miniprojeto tem dados, persistência, interface gráfica e, agora, rotinas de cálculo. Falta transformar esses números em **compreensão visual** — tabelas, séries temporais, gráficos que apoiem a decisão. É o tema do Módulo 4, com o pandas e o Matplotlib.

## 7.7 O caminho à frente

O NumPy nos deu o cálculo eficiente sobre arrays. O **Módulo 4** dá o passo seguinte: o Capítulo 8 apresenta o **pandas**, construído sobre o NumPy, para manipular dados tabulares e séries temporais com rótulos e nomes de colunas; e o **Matplotlib**, para transformar esses dados em gráficos. É a etapa que converte números em decisões — e prepara o Projeto Final.

## 7.8 Resumo do capítulo
- O **array** (`np.array`) é a estrutura central do NumPy: elementos do mesmo tipo, descritos por `shape` e `dtype`.
- As **operações vetorizadas** aplicam um cálculo ao array inteiro sem laços — mais legíveis e muito mais rápidas; *funções universais* como `np.sqrt` e `np.sin` operam da mesma forma.
- As **máscaras booleanas** (`leituras[leituras > limite]`) filtram dados de modo direto e vetorizado.
- As **agregações** (`mean`, `std`, `max`, `np.percentile`) resumem os dados; desvio padrão e percentis caracterizam dispersão e extremos.
- Arrays **bidimensionais** são matrizes; o parâmetro `axis` controla a direção das agregações. A **álgebra linear** (`np.linalg.norm`, `@`, `np.linalg.solve`) sustenta cálculos de distância, transformação e sistemas.
- No **Módulo 3 concluído**, o miniprojeto ganhou rotinas de cálculo: estatísticas das velocidades e detecção de contatos atípicos pela regra "média mais dois desvios".

## Armadilhas comuns
- **Voltar a usar laços por hábito.** Antes de escrever um laço sobre um array, procure a operação vetorizada equivalente — quase sempre existe, e é mais rápida.
- **Misturar tipos em um array.** O array exige um único `dtype`; ao misturar números e texto, o NumPy converte tudo para texto, silenciosamente.
- **Confundir os eixos.** `axis=0` agrega ao longo das linhas (por coluna) e `axis=1`, ao longo das colunas (por linha).
- **Esperar que fatias sejam cópias.** Uma fatia de um array NumPy é uma *vista* dos mesmos dados; alterá-la altera o original. Use `.copy()` se precisar de uma cópia independente.
- **Comparar `shape` incompatíveis.** Operar entre arrays de formas incompatíveis gera erro; verifique o `shape` quando algo não fechar.

Vale ver a armadilha da **vista** (*view*) em ação — é sutil e causa bugs difíceis:

In [ ]:
import numpy as np

original = np.array([10, 20, 30, 40])
fatia = original[1:3]      # uma VISTA, não uma cópia
fatia[0] = 999             # altera a fatia...
print(original)            # [ 10 999  30  40]  -> ...e o original mudou!

copia = original[1:3].copy()   # agora sim, independente
copia[0] = -1
print(original)            # [ 10 999  30  40]  -> original intacto

## Exercícios

### Essencial — fixação
**Ex. 7.1** Crie um array com dez leituras de velocidade à sua escolha. Imprima a forma (`shape`), a média, o valor máximo e o mínimo.

In [ ]:
# Ex. 7.1
import numpy as np

leituras = np.array([28.4, 33.1, 41.7, 39.0, 45.2, 22.0, 51.3, 30.0, 47.8, 19.5])
print("Forma:", leituras.shape)
print("Média:", round(leituras.mean(), 2))
print("Máximo:", leituras.max())
print("Mínimo:", leituras.min())

**Ex. 7.2** A partir de um array de velocidades em nós, obtenha — de forma vetorizada, sem laço — o array em km/h (multiplique por 1,852) e imprima o resultado arredondado a uma casa decimal com `np.round`.

In [ ]:
# Ex. 7.2
import numpy as np

nos = np.array([12, 18, 24, 9, 27])
kmh = nos * 1.852
print(np.round(kmh, 1))     # [22.2 33.3 44.4 16.7 50. ]

**Ex. 7.3** Dado `leituras = np.array([31, 47, 22, 55, 40, 38])`, use uma **máscara booleana** para imprimir apenas as leituras maiores que 40 e, em seguida, quantas são.

In [ ]:
# Ex. 7.3
import numpy as np

leituras = np.array([31, 47, 22, 55, 40, 38])
acima = leituras[leituras > 40]
print("Acima de 40:", acima)          # [47 55]
print("Quantas:", acima.size)         # 2

### Tático — aplicação
**Ex. 7.4** Escreva `resumo(leituras)` que receba um array e devolva um dicionário com a média, o desvio padrão e o percentil 95. Teste com um array de sua escolha.

In [ ]:
# Ex. 7.4
import numpy as np

def resumo(leituras):
    return {
        "media": leituras.mean(),
        "desvio": leituras.std(),
        "p95": np.percentile(leituras, 95),
    }

r = resumo(np.array([28.4, 33.1, 41.7, 39.0, 45.2]))
print({k: round(v, 2) for k, v in r.items()})

**Ex. 7.5** Crie um array bidimensional 3×4 (três sensores ao longo de quatro instantes). Calcule a média de cada sensor (por linha) e a média em cada instante (por coluna), usando `axis`.

In [ ]:
# Ex. 7.5
import numpy as np

M = np.array([
    [28.4, 33.1, 41.7, 39.0],   # sensor 1
    [22.0, 25.5, 30.1, 28.8],   # sensor 2
    [50.0, 48.2, 51.3, 47.9],   # sensor 3
])

print("Média por sensor (axis=1):", np.round(M.mean(axis=1), 2))
print("Média por instante (axis=0):", np.round(M.mean(axis=0), 2))

**Ex. 7.6** Dadas duas posições no plano (arrays de duas coordenadas), escreva `distancia(p, q)` que devolva a distância entre elas com `np.linalg.norm`. Teste com $(0,0)$ e $(3,4)$.

In [ ]:
# Ex. 7.6
import numpy as np

def distancia(p, q):
    return np.linalg.norm(np.array(q) - np.array(p))

print(distancia(np.array([0.0, 0.0]), np.array([3.0, 4.0])))   # 5.0

### Estratégico — extensão criativa
**Ex. 7.7** Estenda as rotinas do miniprojeto (Listagem 7.11) com um método `normalizar` que devolva as velocidades em **escore-z** (subtraída a média, dividida pelo desvio). Reflita sobre por que essa transformação é útil para comparar grandezas de naturezas diferentes — algo recorrente em *Machine Learning*.

*(Reutiliza `RegistroDeOcorrencias` da Listagem 7.11 — execute aquela célula antes.)*

In [ ]:
# Ex. 7.7
import numpy as np

class RegistroComEscoreZ(RegistroDeOcorrencias):
    def normalizar(self):
        """Velocidades em escore-z: (x - média) / desvio."""
        v = self.velocidades()
        if len(v) == 0 or v.std() == 0:
            return v
        return (v - v.mean()) / v.std()

reg = RegistroComEscoreZ()
for sensor, vel in [("Radar-A1", 22.2), ("Radar-A1", 44.4), ("Sonar-1", 18.5),
                    ("Radar-B2", 50.0), ("Radar-B2", 33.1), ("Radar-A1", 120.0)]:
    reg.registrar(sensor, vel)

z = reg.normalizar()
print(np.round(z, 2))
# O contato atípico (120 km/h) tem o maior escore-z, bem acima dos demais.
print("Maior escore-z:", round(z.max(), 2))

**Ex. 7.8** Implemente, **sem** usar a função pronta, o cálculo do desvio padrão: subtraia a média de cada elemento, eleve ao quadrado, tire a média desses quadrados e, por fim, a raiz. Compare com `leituras.std()`.

In [ ]:
# Ex. 7.8
import numpy as np

leituras = np.array([28.4, 33.1, 41.7, 39.0, 45.2])

# Passo a passo, tudo vetorizado
desvios = leituras - leituras.mean()
variancia = (desvios ** 2).mean()
desvio_manual = np.sqrt(variancia)

print("Manual:", round(desvio_manual, 4))
print("NumPy: ", round(leituras.std(), 4))
print("Iguais?", np.isclose(desvio_manual, leituras.std()))

---

*Fim do Capítulo 7 e do Módulo 3. No Capítulo 8, o **pandas** e o **Matplotlib** transformam esses números em tabelas e gráficos de apoio à decisão — a abertura do Módulo 4.*